<a href="https://colab.research.google.com/github/JONNY-ME/data.org-Financial-Health-Prediction/blob/main/approach1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Financial Health Prediction Pipeline (Notebook)

Merged pipeline using **LightGBM** with: quantile clipping, FE, missingness indicators, country scaling & percentile, KNN imputer, SMOTE, finance/yes-no ordinal maps, composite scores, FHI rule bin, extra interactions, calibrated classifier, PDF advanced ratios, structural zero handling, and country-specific models.

In [1]:
import gdown
gdown.download_folder(id='1kEkt_m50XIrSFWDouTrV5sZvmDzyXpef', output='data', quiet=True)

['data/Test.csv', 'data/Train.csv']

## 1. Imports and constants

In [2]:
import os
import warnings
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import HistGradientBoostingClassifier, VotingClassifier, \
                            RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
DATA_DIR = "data"
SUBMISSION_PATH = "sub.csv"

## 2. Data loading

In [3]:
def load_data(data_dir: str = None):
    data_dir = data_dir or DATA_DIR
    train_df = pd.read_csv(os.path.join(data_dir, "Train.csv"))
    test_df = pd.read_csv(os.path.join(data_dir, "Test.csv"))
    return train_df, test_df

train_df, test_df = load_data()

## 3. Quantile clipping

In [4]:
FINANCIAL_COLS = ["personal_income", "business_expenses", "business_turnover"]
AGE_COLS = ["owner_age", "business_age_years", "business_age_months"]

def fit_clip_bounds(train, cols, lower_q=0.001, upper_q=0.999):
    bounds = {}
    for col in cols:
        if col not in train.columns:
            continue
        s = train[col]
        if not pd.api.types.is_numeric_dtype(s):
            continue
        s2 = s.dropna()
        if s2.empty:
            continue
        lo = float(s2.quantile(lower_q))
        hi = float(s2.quantile(upper_q))
        if lo > hi:
            lo, hi = hi, lo
        bounds[col] = (lo, hi)
    return bounds

def apply_clip_bounds(df, bounds):
    out = df.copy()
    for col, (lo, hi) in bounds.items():
        if col in out.columns:
            out[col] = out[col].clip(lower=lo, upper=hi)
    return out

clip_cols = [c for c in (FINANCIAL_COLS + ["business_age_years", "business_age_months"]) if c in train_df.columns]
bounds = fit_clip_bounds(train_df, clip_cols, 0.001, 0.999)
train_df = apply_clip_bounds(train_df, bounds)
test_df = apply_clip_bounds(test_df, bounds)

## 4. Finance product / Yes-No ordinal maps

In [5]:
FINANCE_PRODUCT_MAP = {
    "Have now": 3, "Have now ": 3,
    "Used to have but don't have now": 2, "Used to have but don't have now ": 2,
    "Never had": 1,
    "Don't know": np.nan, "Don't Know": np.nan,
}
YESNO_MAP = {
    "Yes": 1, "Yes, always": 1, "Yes, sometimes": 0.5, "No": 0, "0": 0,
    "Don't know or N/A": np.nan, "Don't know": np.nan, "Don't Know": np.nan, "Refused": np.nan,
}
FINANCE_PRODUCT_COLS = [
    "motor_vehicle_insurance", "has_mobile_money", "has_credit_card", "has_loan_account",
    "has_internet_banking", "has_debit_card", "medical_insurance", "funeral_insurance",
    "uses_friends_family_savings", "uses_informal_lender",
]
YESNO_COLS = [
    "attitude_stable_business_environment", "attitude_worried_shutdown", "compliance_income_tax",
    "perception_insurance_doesnt_cover_losses", "perception_cannot_afford_insurance",
    "current_problem_cash_flow", "has_cellphone", "attitude_satisfied_with_achievement",
    "perception_insurance_companies_dont_insure_businesses_like_yours", "perception_insurance_important",
    "has_insurance", "covid_essential_service", "attitude_more_successful_next_year",
    "problem_sourcing_money", "marketing_word_of_mouth", "future_risk_theft_stock",
    "motivation_make_more_money", "offers_credit_to_customers", "keeps_financial_records",
]

def apply_finance_yesno_maps(df):
    out = df.copy()
    for col in FINANCE_PRODUCT_COLS:
        if col in out.columns:
            out[col] = out[col].astype(str).str.strip().map(FINANCE_PRODUCT_MAP)
    for col in YESNO_COLS:
        if col in out.columns:
            out[col] = out[col].astype(str).str.strip().map(YESNO_MAP)
    return out

train_df = apply_finance_yesno_maps(train_df)
test_df = apply_finance_yesno_maps(test_df)

## 5. Structural zero handling (PDF)

In [6]:
STRUCTURAL_ZERO_COLS = ["current_problem_cash_flow"]

def apply_structural_zero_handling(df):
    out = df.copy()
    for col in STRUCTURAL_ZERO_COLS:
        if col not in out.columns:
            continue
        if out[col].dtype == object or out[col].astype(str).str.strip().eq("0").any():
            out[f"is_structural_zero_{col}"] = (out[col].astype(str).str.strip() == "0").astype(int)
        if pd.api.types.is_numeric_dtype(out[col]):
            out.loc[out[col] == 0, col] = np.nan
    return out

train_df = apply_structural_zero_handling(train_df)
test_df = apply_structural_zero_handling(test_df)

## 6. Feature engineering

In [7]:
STATUS_COLS = [
    "has_mobile_money", "has_credit_card", "has_loan_account", "has_internet_banking",
    "has_debit_card", "motor_vehicle_insurance", "medical_insurance", "funeral_insurance",
]
YES_NO_COLS_CANDIDATES = [
    "attitude_stable_business_environment", "attitude_worried_shutdown", "compliance_income_tax",
    "perception_insurance_doesnt_cover_losses", "perception_cannot_afford_insurance",
    "current_problem_cash_flow", "has_cellphone", "attitude_satisfied_with_achievement",
    "perception_insurance_companies_dont_insure_businesses_like_yours", "perception_insurance_important",
    "has_insurance", "covid_essential_service", "attitude_more_successful_next_year",
    "problem_sourcing_money", "marketing_word_of_mouth", "future_risk_theft_stock",
    "motivation_make_more_money",
]

def _status_to_has_now(val):
    if pd.isna(val):
        return np.nan
    if val == "Have now" or val == 3:
        return 1.0
    if val in {"Never had", "Used to have"} or val in (1, 2):
        return 0.0 if val == "Never had" or val == 1 else 0.5
    return np.nan

def _yesno_to_binary(val):
    if pd.isna(val):
        return np.nan
    if val == "Yes" or val == 1:
        return 1.0
    if val == "No" or val == 0:
        return 0.0
    if val == 0.5:
        return 0.5
    return np.nan

def engineer_features(df):
    out = df.copy()
    if "business_turnover" in out.columns and "business_expenses" in out.columns:
        out["gross_margin"] = out["business_turnover"] - out["business_expenses"]
        out["expense_to_turnover"] = out["business_expenses"] / (out["business_turnover"] + 1e-6)
    if "business_age_years" in out.columns and "business_age_months" in out.columns:
        out["age_fraction"] = out["business_age_years"] + out["business_age_months"] / 12.0
    ins_cols = [c for c in out.columns if c.endswith("insurance") and c != "perception_insurance_doesnt_cover_losses"]
    if ins_cols:
        def _ins_one(x):
            if pd.isna(x):
                return 0
            s = str(x).strip().lower()
            if s in ("yes", "have now"):
                return 1
            if isinstance(x, (int, float)) and x == 3:
                return 1
            return 0
        out["insurance_count"] = out[ins_cols].apply(lambda col: col.map(_ins_one)).sum(axis=1)
    debt_cols = [c for c in ["has_loan_account", "has_credit_card", "uses_informal_lender"] if c in out.columns]
    if debt_cols:
        formal_s = out.get("has_loan_account", pd.Series(0, index=out.index))
        out["formal_debt"] = (formal_s >= 2).astype(int).fillna(0) if formal_s.dtype != object else formal_s.map(lambda x: 1 if str(x).strip().lower() == "yes" else 0).fillna(0)
        informal_s = out.get("uses_informal_lender", pd.Series(0, index=out.index))
        out["informal_debt"] = (informal_s >= 2).astype(int).fillna(0) if informal_s.dtype != object else informal_s.map(lambda x: 1 if str(x).strip().lower() == "yes" else 0).fillna(0)
        out["debt_indicator"] = out["formal_debt"].astype(float) + out["informal_debt"].astype(float)
    if "owner_age" in out.columns:
        out["owner_age_sq"] = out["owner_age"] ** 2
        out["owner_age_log"] = np.log1p(out["owner_age"])
    attitude_vars = ["attitude_stable_business_environment", "attitude_worried_shutdown", "attitude_satisfied_with_achievement", "attitude_more_successful_next_year"]
    for var in attitude_vars:
        if var in out.columns:
            v = out[var]
            out[var + "_enc"] = out[var].map({"Yes": 1, "No": 0}) if v.dtype == object else v
    years = out.get("business_age_years", pd.Series(0, index=out.index)).fillna(0)
    months = out.get("business_age_months", pd.Series(0, index=out.index)).fillna(0)
    out["fe_business_age_total_months"] = years * 12.0 + months
    out["fe_business_age_total_years"] = out["fe_business_age_total_months"] / 12.0
    turnover, expenses, income = out.get("business_turnover"), out.get("business_expenses"), out.get("personal_income")
    if turnover is not None and expenses is not None:
        out["fe_profit"] = turnover - expenses
        out["fe_expense_ratio"] = expenses / (turnover + 1.0)
        out["fe_profit_margin"] = out["fe_profit"] / (turnover + 1.0)
    if income is not None and turnover is not None:
        out["fe_income_to_turnover"] = income / (turnover + 1.0)
    if turnover is not None:
        out["fe_turnover_per_business_year"] = turnover / (out.get("fe_business_age_total_years", 0) + 0.5)
    if expenses is not None:
        out["fe_expenses_per_business_year"] = expenses / (out.get("fe_business_age_total_years", 0) + 0.5)
    for col in ["personal_income", "business_expenses", "business_turnover", "fe_profit"]:
        if col in out.columns:
            out[f"fe_log1p_{col}"] = np.log1p(out[col].clip(lower=0))
    if "owner_age" in out.columns:
        out["fe_owner_age_minus_business_age_years"] = out["owner_age"] - out.get("fe_business_age_total_years", 0)
    for col in YES_NO_COLS_CANDIDATES:
        if col in out.columns:
            out[f"fe_{col}_bin"] = out[col].map(_yesno_to_binary) if out[col].dtype == object else out[col]
    for col in STATUS_COLS:
        if col in out.columns:
            out[f"fe_{col}_has_now"] = out[col].map(_status_to_has_now) if out[col].dtype == object else (out[col] == 3).astype(float)
    if "offers_credit_to_customers" in out.columns:
        out["fe_offers_credit_level"] = out["offers_credit_to_customers"].map({"No": 0, "Yes, sometimes": 1, "Yes, always": 2}) if out["offers_credit_to_customers"].dtype == object else out["offers_credit_to_customers"]
    if "keeps_financial_records" in out.columns:
        out["fe_keeps_financial_records_level"] = out["keeps_financial_records"].map({"No": 0, "Yes": 1, "Yes, sometimes": 1, "Yes, always": 2}) if out["keeps_financial_records"].dtype == object else out["keeps_financial_records"]
    now_cols = [f"fe_{c}_has_now" for c in STATUS_COLS if f"fe_{c}_has_now" in out.columns]
    if now_cols:
        out["fe_num_products_have_now"] = out[now_cols].sum(axis=1, min_count=1)
    ins_now = [c for c in ["fe_motor_vehicle_insurance_has_now", "fe_medical_insurance_has_now", "fe_funeral_insurance_has_now"] if c in out.columns]
    if ins_now:
        out["fe_num_insurance_have_now"] = out[ins_now].sum(axis=1, min_count=1)
    pos_cols = [c for c in ["fe_attitude_stable_business_environment_bin", "fe_attitude_satisfied_with_achievement_bin", "fe_attitude_more_successful_next_year_bin"] if c in out.columns]
    if pos_cols:
        out["fe_num_positive_attitudes"] = out[pos_cols].sum(axis=1, min_count=1)
    def _map_access(val):
        if pd.isna(val):
            return np.nan
        if val in (3, "Have now", "Yes"):
            return 1.0
        if val in (2, "Used to have"):
            return 0.5
        return 0.0
    formal = [c for c in ["has_loan_account", "has_internet_banking", "has_debit_card", "medical_insurance", "funeral_insurance", "has_mobile_money", "has_credit_card", "has_insurance"] if c in out.columns]
    if formal:
        scores = out[formal].apply(lambda col: col.map(_map_access) if col.dtype == object else col.replace(3, 1).replace(2, 0.5).replace(1, 0))
        n_valid = scores.notna().sum(axis=1)
        out["financial_access_score"] = np.where(n_valid > 0, scores.sum(axis=1) / n_valid, np.nan)
    informal = [c for c in ["uses_friends_family_savings", "uses_informal_lender"] if c in out.columns]
    if informal:
        scores = out[informal].apply(lambda col: col.map(_map_access) if col.dtype == object else col.replace(3, 1).replace(2, 0.5).replace(1, 0))
        n_valid = scores.notna().sum(axis=1)
        out["informal_access_score"] = np.where(n_valid > 0, scores.sum(axis=1) / n_valid, np.nan)
    stress = pd.Series(0.0, index=out.index)
    for col, check in [("attitude_worried_shutdown", lambda s: s.fillna("").astype(str).str.strip().str.lower() == "yes"), ("current_problem_cash_flow", lambda s: s.fillna("").astype(str).str.strip().str.lower() == "yes"), ("problem_sourcing_money", lambda s: s.fillna("").astype(str).str.strip().str.lower() == "yes")]:
        if col in out.columns:
            stress += check(out[col]).astype(float)
    out["stress_score"] = stress
    if "owner_age" in out.columns:
        mat = out.get("fe_business_age_total_years", out.get("age_fraction", pd.Series(0, index=out.index)))
        out["owner_business_age_interaction"] = out["owner_age"] * mat
    if "stress_score" in out.columns and "expense_to_turnover" in out.columns:
        out["stress_x_expense_ratio"] = out["stress_score"] * out["expense_to_turnover"].clip(0, 5)
    if "financial_access_score" in out.columns and "business_turnover" in out.columns:
        out["access_x_log_turnover"] = out["financial_access_score"].fillna(0) * np.log1p(out["business_turnover"].clip(lower=0).fillna(0))
    if "owner_age" in out.columns and "fe_num_products_have_now" in out.columns:
        out["owner_age_x_formal_products"] = out["owner_age"].fillna(0) * out["fe_num_products_have_now"].fillna(0)
    acc = out.get("financial_access_score", pd.Series(0, index=out.index)).fillna(0)
    stress = out.get("stress_score", pd.Series(0, index=out.index)).fillna(0)
    pm = out.get("fe_profit_margin", pd.Series(0, index=out.index)).fillna(0)
    if "expense_to_turnover" in out.columns:
        etr = out["expense_to_turnover"].fillna(0).clip(0, 1)
        profit_ok = (1 - etr) >= 0.1
    else:
        profit_ok = pm >= 0.1
    high = (acc >= 0.5) & (stress <= 1) & profit_ok
    medium = ((acc >= 0.3) & (stress <= 2)) | ((acc >= 0.2) & (stress <= 1))
    rule_bin = np.zeros(len(out), dtype=int)
    rule_bin[medium & ~high] = 1
    rule_bin[high] = 2
    out["fhi_rule_bin"] = rule_bin
    if "business_turnover" in out.columns and "business_expenses" in out.columns:
        turnover = out["business_turnover"].fillna(0) + 1e-9
        out["operational_profitability_margin"] = ((out["business_turnover"] - out["business_expenses"]) / turnover).clip(-1, 1)
    formal_acc = out.get("financial_access_score", pd.Series(0, index=out.index)).fillna(0)
    informal_acc = out.get("informal_access_score", pd.Series(0, index=out.index)).fillna(0)
    out["credit_access_vulnerability"] = (1 - formal_acc) * (0.5 + informal_acc)
    return out

train_fe = engineer_features(train_df)
test_fe = engineer_features(test_df)

## 7. Country scaling and country percentile

In [8]:
def fit_country_scale_params(train, financial_cols):
    params = {}
    for col in financial_cols:
        if col not in train.columns or "country" not in train.columns:
            continue
        t = train.copy()
        t["_log1p_"] = np.log1p(t[col].clip(lower=0).fillna(0))
        grp = t.groupby("country")["_log1p_"].agg(["mean", "std"])
        grp["std"] = grp["std"].replace(0, np.nan).fillna(1.0)
        params[col] = grp.to_dict("index")
    return params

def add_country_scaled_features(df, params, financial_cols):
    out = df.copy()
    if "country" not in out.columns:
        return out
    for col in financial_cols:
        if col not in params:
            continue
        log_vals = np.log1p(out[col].clip(lower=0).fillna(0))
        out[f"{col}_country_scaled"] = np.nan
        for country, stats in params[col].items():
            mask = out["country"].astype(str).str.lower() == str(country).lower()
            out.loc[mask, f"{col}_country_scaled"] = (log_vals.loc[mask] - stats["mean"]) / (stats["std"] + 1e-9)
    return out

from scipy.stats import percentileofscore

def fit_country_percentile_params(train, financial_cols):
    params = {}
    for col in financial_cols:
        if col not in train.columns or "country" not in train.columns:
            continue
        params[col] = {}
        for country in train["country"].dropna().unique():
            mask = train["country"] == country
            vals = train.loc[mask, col].dropna().values
            params[col][str(country)] = vals if len(vals) > 0 else None
    return params

def add_country_percentile_features(df, params, financial_cols, is_train=False):
    out = df.copy()
    if "country" not in out.columns:
        return out
    for col in financial_cols:
        if col not in params:
            continue
        out[f"{col}_country_pct"] = np.nan
        for country, train_vals in params[col].items():
            if train_vals is None or len(train_vals) == 0:
                continue
            train_vals = np.asarray(train_vals, dtype=float)
            mask = out["country"].astype(str) == str(country)
            if is_train:
                out.loc[mask, f"{col}_country_pct"] = out.loc[mask, col].rank(pct=True).values
            else:
                for idx in out.index[mask]:
                    v = out.loc[idx, col]
                    if pd.isna(v):
                        continue
                    out.loc[idx, f"{col}_country_pct"] = percentileofscore(train_vals, float(v), kind="rank") / 100.0
    return out

fin_cols = [c for c in FINANCIAL_COLS if c in train_fe.columns]
country_params = fit_country_scale_params(train_fe, fin_cols)
train_fe = add_country_scaled_features(train_fe, country_params, fin_cols)
test_fe = add_country_scaled_features(test_fe, country_params, fin_cols)
country_pct_params = fit_country_percentile_params(train_fe, fin_cols)
train_fe = add_country_percentile_features(train_fe, country_pct_params, fin_cols, is_train=True)
test_fe = add_country_percentile_features(test_fe, country_pct_params, fin_cols, is_train=False)

## 8. Missingness indicators and feature matrix

In [9]:
def add_missingness_indicators(df, feature_cols):
    out = df.copy()
    for col in feature_cols:
        if col in out.columns:
            out[f"fe_is_missing_{col}"] = out[col].isna().astype(int)
    return out

X = train_fe.drop(columns=["ID", "Target"])
y = train_fe["Target"]
feature_cols = X.columns.tolist()
train_fe = add_missingness_indicators(train_fe, feature_cols)
test_fe = add_missingness_indicators(test_fe, feature_cols)
X = train_fe.drop(columns=["ID", "Target"])
feature_cols = X.columns.tolist()

## 9. Preprocessor (KNN imputer, OneHot for categoricals)

In [10]:
def build_preprocessor(train_df):
    X = train_df.drop(columns=["ID", "Target"], errors="ignore")
    numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    imputer = KNNImputer(n_neighbors=5)
    numeric_pipeline = Pipeline(steps=[("imputer", imputer), ("scaler", StandardScaler())])
    cat_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="<MISSING>")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    preprocessor = ColumnTransformer(
        transformers=[("num", numeric_pipeline, numeric_cols), ("cat", cat_pipeline, cat_cols)],
        remainder="drop",
    )
    return preprocessor

preprocessor = build_preprocessor(train_fe)

## 10. Evaluation and LGB hyperparameter tuning

In [11]:
LGB_BEST_PARAMS_OVERRIDE = {
    'n_estimators': 494, 'max_depth': 6, 'num_leaves': 136,
    'min_child_samples': 86, 'learning_rate': 0.019103905921489524,
    'reg_lambda': 0.009360997474964425
}
XGB_BEST_PARAMS_OVERRIDE = {
    'n_estimators': 534, 'max_depth': 5, 'learning_rate': 0.010074139715957057,
    'subsample': 0.7264385884072835, 'colsample_bytree': 0.6306650799985789,
    'reg_alpha': 1.6448392892048587, 'reg_lambda': 1.7566924660016257
}
HGB_BEST_PARAMS_OVERRIDE = {
    'max_iter': 328, 'max_depth': 10, 'min_samples_leaf': 24,
    'learning_rate': 0.018785426399210624,
    'l2_regularization': 0.09163741808778776, 'max_bins': 133
}

def evaluate_model(estimator, X, y, cv=5, random_state=RANDOM_STATE):
    cv_split = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    splits = list(cv_split.split(X, y))
    scores = []
    for train_idx, val_idx in splits:
        estimator.fit(X.iloc[train_idx], y[train_idx])
        pred = estimator.predict(X.iloc[val_idx])
        scores.append(f1_score(y[val_idx], pred, average="macro"))
    return np.mean(scores), np.std(scores)


## 11. Pipeline: Calibrated LGB + SMOTE

In [12]:
base_clf = VotingClassifier(estimators=[
    ("lgb", LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, **LGB_BEST_PARAMS_OVERRIDE)),
    ("xgb", XGBClassifier(random_state=RANDOM_STATE, verbosity=0, **XGB_BEST_PARAMS_OVERRIDE)),
    ("hgb", HistGradientBoostingClassifier(random_state=RANDOM_STATE)),
    # ("rf", RandomForestClassifier(n_estimators=350, random_state=RANDOM_STATE, class_weight='balanced')),
    # ("lrg", LogisticRegression(max_iter=400, random_state=RANDOM_STATE)),
], voting="soft")
base_clf = CalibratedClassifierCV(base_clf, cv=3, method="isotonic")
pipe = ImbPipeline([
    ("pre", preprocessor),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("clf", base_clf),
])

# mean_f1, std_f1 = evaluate_model(pipe, X, y)
# print(f"LGB (Calibrated + SMOTE) CV F1_macro = {mean_f1:.4f} ± {std_f1:.4f}")

## 12. Train and country-specific models

In [13]:
pipe.fit(X, y)

country_pipes = {}
for c in X["country"].dropna().unique():
    mask = X["country"] == c
    X_c = X.loc[mask]
    y_c = y.loc[mask]
    if len(y_c) < 15 or y_c.nunique() < 2:
        country_pipes[c] = pipe
    else:
        pipe_c = clone(pipe)
        pipe_c.fit(X_c, y_c)
        country_pipes[c] = pipe_c
print(f"Country-specific models: {len([c for c in country_pipes if country_pipes[c] is not pipe])} trained.")

Country-specific models: 4 trained.


## 13. Predict and save submission

In [14]:
test_X = test_fe.drop(columns=["ID"], errors="ignore")
test_X = test_X.reindex(columns=X.columns, fill_value=np.nan)
proba = pipe.predict_proba(test_X)
classes = np.asarray(pipe.named_steps["clf"].classes_)
test_countries = test_X["country"].values
for c in np.unique(test_countries):
    pipe_c = country_pipes.get(c, pipe)
    pos_mask = np.where(test_countries == c)[0]
    proba[pos_mask] = pipe_c.predict_proba(test_X.iloc[pos_mask])
labels = classes[np.argmax(proba, axis=1)]
submission = pd.DataFrame({"ID": test_fe["ID"].values, "Target": labels})
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Submission written to {SUBMISSION_PATH}")

Submission written to sub.csv


In [15]:
from google.colab import files
files.download(SUBMISSION_PATH)
submission.Target.value_counts()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,count
Target,
Low,1650
Medium,659
High,96
